In [94]:
import duckdb
from pathlib import Path
from data_importers import LandRegistryDataImporter, OnsPostcodeDirectoryImporter

## Download data from Land Registry

In [95]:
DATA_PATH = Path("../../data/land_registry/")
assert DATA_PATH.parent.exists(), f"Data path {DATA_PATH} does not exist. Please update the path to point to the correct location of the land registry data."

In [96]:
land_registry_data = LandRegistryDataImporter(DATA_PATH)

In [97]:
# Note - this operation downloads ~150MB per year, so may take a while to run
land_registry_data.download_land_registry_data(2023, 2026)

In [98]:
ONS_PD_DATA_PATH = Path("../../data/ons_postcode_directory/")

In [99]:
ons_postcode_directory_data = OnsPostcodeDirectoryImporter(ONS_PD_DATA_PATH)

In [100]:
# Note - this operation downloads ~1.5GB of data, so may take a while to run
ons_postcode_directory_data.import_data()

## Set up a database and load the raw data into a table

We're going to create a DuckDB database to persist the data we have extracted from the CSV files and then take it through a few transformation steps to finally generate some useful analytics over the data.

In [101]:
from pathlib import Path
LAND_REGISTRY_DATABASE_PATH = Path("../../databases/land_registry.db")

In [102]:
DELETE_EXISTING_DATABASE = False

In [103]:
if LAND_REGISTRY_DATABASE_PATH.exists() and DELETE_EXISTING_DATABASE:
    LAND_REGISTRY_DATABASE_PATH.unlink()
else:
    LAND_REGISTRY_DATABASE_PATH.parent.mkdir(parents=True, exist_ok=True)

In [104]:
db = duckdb.connect(LAND_REGISTRY_DATABASE_PATH)

In [105]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬─────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │               table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                 │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼─────────────────────────────────────────┼────────────┼──────────────────────────────┼──────────────────────┼─

## Load raw data into a table

### Price Paid Data

In [106]:
db.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [107]:
db.sql(
    f"""
    CREATE TABLE IF NOT EXISTS bronze.price_paid_raw AS
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    FROM read_csv('{DATA_PATH / "pp-*.csv"}');
    """
)

In [108]:
db.sql("SELECT COUNT (*) FROM bronze.price_paid_raw")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      2739865 │
└──────────────┘

In [109]:
db.sql("FROM bronze.price_paid_raw LIMIT 5")

┌────────────────────────────────────────┬────────┬─────────────────────┬──────────┬───────────────┬─────────┬──────────┬─────────┬─────────┬─────────────────┬───────────┬───────────┬──────────────────┬──────────────────┬──────────────┬─────────────┐
│                   id                   │ price  │        date         │ postcode │ property_type │ old_new │ duration │  paon   │  saon   │     street      │  locale   │ town_city │     district     │      county      │ ppd_category │ record_type │
│                varchar                 │ int64  │      timestamp      │ varchar  │    varchar    │ varchar │ varchar  │ varchar │ varchar │     varchar     │  varchar  │  varchar  │     varchar      │     varchar      │   varchar    │   varchar   │
├────────────────────────────────────────┼────────┼─────────────────────┼──────────┼───────────────┼─────────┼──────────┼─────────┼─────────┼─────────────────┼───────────┼───────────┼──────────────────┼──────────────────┼──────────────┼───────────

### ONS Postcode Directory

In [154]:
db.sql(
    f"""
    CREATE TABLE IF NOT EXISTS bronze.postcode_directory AS
    SELECT pcd8 AS 'postcode',
    lat as 'latitude',
    long as 'longitude'
    FROM read_csv('{ONS_PD_DATA_PATH / "*.csv"}', header=true, quote='"');
    """
)

In [156]:
db.sql("SELECT COUNT (*) FROM bronze.postcode_directory")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      2723596 │
└──────────────┘

In [157]:
db.sql("FROM bronze.postcode_directory LIMIT 5")

┌──────────┬───────────┬───────────┐
│ postcode │ latitude  │ longitude │
│ varchar  │  double   │  double   │
├──────────┼───────────┼───────────┤
│ AB1 0AA  │ 57.101459 │ -2.242858 │
│ AB1 0AB  │ 57.102539 │ -2.246315 │
│ AB1 0AD  │ 57.100541 │ -2.248349 │
│ AB1 0AE  │ 57.084429 │ -2.255714 │
│ AB1 0AF  │ 57.096641 │ -2.258109 │
└──────────┴───────────┴───────────┘

In [158]:
db.sql("DESCRIBE bronze.postcode_directory")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ postcode    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latitude    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ longitude   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

## Add features

In [159]:
db.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [160]:
db.sql(
    r"""
    CREATE OR REPLACE VIEW silver.price_paid_cleaned AS
    SELECT
    id,
    price,
    date,
    year(date) AS 'year_of_sale',
    date_trunc('month', date) AS 'month_of_sale',
    CASE property_type
          WHEN 'D' THEN 'Detached'
          WHEN 'S' THEN 'Semi-Detached'
          WHEN 'T' THEN 'Terraced'
          WHEN 'F' THEN 'Flat'
          WHEN 'O' THEN 'Other'
          ELSE property_type
    END AS 'property_type',
    postcode,
    paon,
    saon,
    street,
    locale,
    town_city,
    district,
    county,
    FROM bronze.price_paid_raw
    WHERE property_type <> 'O';
    """
)

In [161]:
db.sql(
    """
    SELECT
    * 
    FROM silver.price_paid_cleaned LIMIT 10
    """
)

┌────────────────────────────────────────┬────────┬─────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────┬─────────┬─────────┬─────────────────┬───────────┬──────────────┬──────────────────┬──────────────────┐
│                   id                   │ price  │        date         │ year_of_sale │    month_of_sale    │ property_type │ postcode │  paon   │  saon   │     street      │  locale   │  town_city   │     district     │      county      │
│                varchar                 │ int64  │      timestamp      │    int64     │      timestamp      │    varchar    │ varchar  │ varchar │ varchar │     varchar     │  varchar  │   varchar    │     varchar      │     varchar      │
├────────────────────────────────────────┼────────┼─────────────────────┼──────────────┼─────────────────────┼───────────────┼──────────┼─────────┼─────────┼─────────────────┼───────────┼──────────────┼──────────────────┼──────────────────┤
│ {0E082196-CE18-5C09-E063-4704A8C0A

In [170]:
db.sql(
    r"""
    CREATE OR REPLACE VIEW silver.postcodes AS
    SELECT
    postcode AS 'postcode_raw',
    longitude,
    latitude,
    NULLIF(
        regexp_extract(postcode, '^([A-Z]{1,2})[0-9]{1,2}[A-Z]?\s*?[0-9][A-Z]{2}$', 1),
        ''
    ) AS postcode_area,
    NULLIF(
        regexp_extract(postcode, '^([A-Z]{1,2}[0-9]{1,2})[A-Z]?\s*?[0-9][A-Z]{2}$', 1),
        ''
    ) AS postcode_district,
    NULLIF(
        regexp_extract(postcode, '^([A-Z]{1,2}[0-9]{1,2}[A-Z]?\s*?[0-9])[A-Z]{2}$', 1),
        ''
    ) AS postcode_sector,
    NULLIF(
        regexp_extract(postcode, '^([A-Z]{1,2}[0-9]{1,2}[A-Z]?)\s*?[0-9][A-Z]{2}$', 1),
        ''
    ) AS postcode_outward_code,
    NULLIF(
        regexp_extract(postcode, '^[A-Z]{1,2}[0-9]{1,2}[A-Z]?\s*?([0-9][A-Z]{2})$', 1),
        ''
    ) AS postcode_inward_code,
    CONCAT(postcode_outward_code, ' ', postcode_inward_code) AS 'postcode'
    FROM bronze.postcode_directory;
    """
)

In [171]:
db.sql(
    """
    FROM silver.postcodes LIMIT 10
    """
)

┌──────────────┬───────────┬───────────┬───────────────┬───────────────────┬─────────────────┬───────────────────────┬──────────────────────┬──────────┐
│ postcode_raw │ longitude │ latitude  │ postcode_area │ postcode_district │ postcode_sector │ postcode_outward_code │ postcode_inward_code │ postcode │
│   varchar    │  double   │  double   │    varchar    │      varchar      │     varchar     │        varchar        │       varchar        │ varchar  │
├──────────────┼───────────┼───────────┼───────────────┼───────────────────┼─────────────────┼───────────────────────┼──────────────────────┼──────────┤
│ AB1 0AA      │ -2.242858 │ 57.101459 │ AB            │ AB1               │ AB1 0           │ AB1                   │ 0AA                  │ AB1 0AA  │
│ AB1 0AB      │ -2.246315 │ 57.102539 │ AB            │ AB1               │ AB1 0           │ AB1                   │ 0AB                  │ AB1 0AB  │
│ AB1 0AD      │ -2.248349 │ 57.100541 │ AB            │ AB1               │ AB1 0

In [172]:
db.sql(
    """
    FROM silver.postcodes
    WHERE postcode_area IS NULL
    LIMIT 10
    """
)

┌──────────────┬───────────┬───────────┬───────────────┬───────────────────┬─────────────────┬───────────────────────┬──────────────────────┬──────────┐
│ postcode_raw │ longitude │ latitude  │ postcode_area │ postcode_district │ postcode_sector │ postcode_outward_code │ postcode_inward_code │ postcode │
│   varchar    │  double   │  double   │    varchar    │      varchar      │     varchar     │        varchar        │       varchar        │ varchar  │
├──────────────┼───────────┼───────────┼───────────────┼───────────────────┼─────────────────┼───────────────────────┼──────────────────────┼──────────┤
│ GIR 0AA      │       0.0 │ 99.999999 │ NULL          │ NULL              │ NULL            │ NULL                  │ NULL                 │          │
│ NPT 0AD      │ -2.991593 │ 51.588461 │ NULL          │ NULL              │ NULL            │ NULL                  │ NULL                 │          │
│ NPT 0AE      │ -2.991632 │ 51.590259 │ NULL          │ NULL              │ NULL 

## Checking for orphaned properties

In [173]:
# Join price paid data with postcode data to get longitude and latitude for each property sale
db.sql(
    r"""
    CREATE OR REPLACE VIEW silver.price_paid_geocoded AS
    SELECT
    p.id,
    p.price,
    p.date,
    p.year_of_sale,
    p.month_of_sale,
    p.property_type,
    p.postcode,
    p.paon,
    p.saon,
    p.street,
    p.locale,
    p.town_city,
    p.district,
    p.county,
    s.longitude,
    s.latitude
    FROM silver.price_paid_cleaned p
    LEFT JOIN silver.postcodes s
        ON p.postcode = s.postcode;
    """
)

In [174]:
db.sql(
    """
    FROM silver.price_paid_geocoded
    WHERE longitude IS NULL
    """
)

┌────────────────────────────────────────┬─────────┬─────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────┬──────────────────────────────┬───────────┬──────────────────────┬───────────────────────┬─────────────────────┬──────────────────────────┬──────────────────────────┬───────────┬──────────┐
│                   id                   │  price  │        date         │ year_of_sale │    month_of_sale    │ property_type │ postcode │             paon             │   saon    │        street        │        locale         │      town_city      │         district         │          county          │ longitude │ latitude │
│                varchar                 │  int64  │      timestamp      │    int64     │      timestamp      │    varchar    │ varchar  │           varchar            │  varchar  │       varchar        │        varchar        │       varchar       │         varchar          │         varchar          │  double   │  double  │
├───────────────

## Projecting insights

### Monthly statistics by property type

In [79]:
db.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [80]:
db.sql(
    """
    CREATE OR REPLACE VIEW gold.median_price_by_month_and_property_type AS
    SELECT
    property_type,
    month_of_sale,
    MEDIAN(price) AS median_price
    FROM silver.price_paid_cleaned
    GROUP BY ALL
    ORDER BY month_of_sale, property_type;
    """
)

In [81]:
db.sql("FROM gold.median_price_by_month_and_property_type")

┌───────────────┬─────────────────────┬──────────────┐
│ property_type │    month_of_sale    │ median_price │
│    varchar    │      timestamp      │    double    │
├───────────────┼─────────────────────┼──────────────┤
│ Detached      │ 2023-01-01 00:00:00 │     430000.0 │
│ Flat          │ 2023-01-01 00:00:00 │     238000.0 │
│ Semi-Detached │ 2023-01-01 00:00:00 │     260000.0 │
│ Terraced      │ 2023-01-01 00:00:00 │     212795.0 │
│ Detached      │ 2023-02-01 00:00:00 │     420000.0 │
│ Flat          │ 2023-02-01 00:00:00 │     230000.0 │
│ Semi-Detached │ 2023-02-01 00:00:00 │     256000.0 │
│ Terraced      │ 2023-02-01 00:00:00 │     212500.0 │
│ Detached      │ 2023-03-01 00:00:00 │     414995.0 │
│ Flat          │ 2023-03-01 00:00:00 │     227744.0 │
│  ·            │          ·          │         ·    │
│  ·            │          ·          │         ·    │
│  ·            │          ·          │         ·    │
│ Semi-Detached │ 2026-01-01 00:00:00 │     270000.0 │
│ Terraced

### Location dimension

In [82]:
db.sql(
    """
    CREATE OR REPLACE VIEW gold.locations AS
    SELECT
    town_city,
    district,
    county
    FROM silver.price_paid_cleaned
    WHERE property_type <> 'Other'
    GROUP BY ALL
    ORDER BY county, district, town_city;
    """
)

In [83]:
db.sql("SELECT COUNT(*) FROM gold.locations")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         2540 │
└──────────────┘

## Inspect contents of database

In [84]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬─────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │               table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                 │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼─────────────────────────────────────────┼────────────┼──────────────────────────────┼──────────────────────┼─

## Inspect the files

In [85]:
database_files = list(LAND_REGISTRY_DATABASE_PATH.parent.glob("*.db"))
csv_files = list(DATA_PATH.glob("*.csv"))

In [86]:
database_files

[PosixPath('../../databases/land_registry.db')]

In [87]:
import polars as pl

files_to_analyse = pl.DataFrame(
    { 
        "file_name": [f.name for f in database_files + csv_files],
        "file_size": [f.stat().st_size for f in database_files + csv_files],
        "file_type": [f.suffix for f in database_files + csv_files]
    }
)

In [88]:
files_to_analyse

file_name,file_size,file_type
str,i64,str
"""land_registry.db""",153104384,""".db"""
"""pp-2025.csv""",147899769,""".csv"""
"""pp-2023.csv""",150073808,""".csv"""
"""pp-2024.csv""",161466225,""".csv"""
"""pp-2026.csv""",18087606,""".csv"""


In [89]:
(
    files_to_analyse
    .group_by("file_type")
    .agg(pl.col("file_size").sum().alias("total_file_size"))
    .with_columns((pl.col("total_file_size") / (1024 * 1024 * 1024)).alias("total_file_size_gb"))
)

file_type,total_file_size,total_file_size_gb
str,i64,f64
""".db""",153104384,0.14259
""".csv""",477527408,0.444732


In [90]:
# Calculate total sizes for .db and .csv files
db_size = files_to_analyse.filter(pl.col("file_type") == ".db")["file_size"].sum()
csv_size = files_to_analyse.filter(pl.col("file_type") == ".csv")["file_size"].sum()

# Calculate compression percentage
compression_percentage = (db_size / csv_size) * 100
print(f"DuckDB database is {compression_percentage:.1f}% of the original size of the CSV files.")

DuckDB database is 32.1% of the original size of the CSV files.


## Add visualisation over the analytics

Finally we convert the Gold `average_price_by_month_and_property_type` view to a Polars dataframe so we can visualise the data on a line chart using the Plotly Express charting package.

In [91]:
import plotly.express as px

fig = px.line(
    db.sql("FROM gold.median_price_by_month_and_property_type").pl(),
    x="month_of_sale",
    y="median_price",
    color="property_type",
    line_dash="property_type",
    color_discrete_sequence=["#8CC600", "black", "darkgrey", "#EE7423"],
    line_dash_sequence=["solid", "dash", "dot", "dashdot"],
    title="Median Price by Month and Property Type"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Median Price",
    legend_title="Property Type",
)

fig.show()

In [92]:
db.close()